In [0]:
import os,sys

In [0]:
sys.path.insert(0,os.path.abspath(os.path.join(os.getcwd(), '..')))

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
from utils.spark_utils import(
    add_silver_audit_columns,
    write_delta_overwrite,
    write_delta_append,
    dedup,
    drop_nulls,
    
)
from config.config import(
    BRONZE_CUSTOMERS,BRONZE_ORDERS,BRONZE_PRODUCTS,
    BRONZE_PATH,SILVER_PATH,GOLD_PATH,
)
from utils.dq_checks import(
    standard_checks,
)

## procesing customers

In [0]:
df_customers=spark.read.format("delta")\
        .option('inferSchema','true')\
        .load('/Volumes/salesdw/bronze/bronze_data/bronze_customers')   



In [0]:
df_customers.display()

In [0]:
df_customers=add_silver_audit_columns(df_customers)

In [0]:
df_customers=dedup(df_customers,['customer_id','email'],'registration_date')

In [0]:
df_customers=drop_nulls(df_customers,['customer_id','email','first_name'])

In [0]:
df_customers.display()

In [0]:
df_customers=df_customers.withColumn('customer_id',trim(upper(col("customer_id")))).withColumn('first_name',initcap(trim(col("first_name")))).withColumn('last_name',initcap(trim(col("last_name")))).withColumn('email',lower(trim(col("email")))).withColumn('phone',regexp_replace(col("phone"),r"[^0-9]",'')).withColumn('city',initcap(trim(col("city")))).withColumn('state',initcap(trim(col("state")))).withColumn('country',initcap(trim(col("country")))).withColumn('zip_code',trim(col('zip_code'))).withColumn('customer_segment',upper(trim(col("customer_segment")))) .withColumn("registration_date", to_date(col("registration_date"), "yyyy-MM-dd")).withColumn("created_at",current_timestamp()).withColumn("updated_at",current_timestamp())


In [0]:
final_cols = [
        "customer_id", "first_name", "last_name",
        "email", "phone", "city", "state", "country", "zip_code",
        "customer_segment", "registration_date",
        "source_system", "batch_id", "ingested_at", "created_at", "updated_at"
]
df_customers=df_customers.select(*final_cols)

**Data Quality Checks**

In [0]:
standard_checks(df_customers,'customers',['email','customer_id','first_name'],['customer_id'])

In [0]:
write_delta_overwrite(df_customers,'/Volumes/salesdw/silver/silver_data/silver_customers',['customer_id'])

## Processing Products

In [0]:
df_products=spark.read.format("delta")\
        .option('inferSchema','true')\
        .load('/Volumes/salesdw/bronze/bronze_data/bronze_products')

In [0]:
df_products=add_silver_audit_columns(df_products)

In [0]:
df_products.display()

In [0]:
df_products=dedup(df_products,['product_id','product_name','category','brand'],'created_date')

In [0]:
df_products=drop_nulls(df_products,['product_id','product_name','category','brand'])
df_products=df_products.withColumn('product_id',trim(upper(col("product_id")))).withColumn('product_name',trim(col("product_name"))).withColumn('category',trim(col("category"))).withColumn('brand',trim(col("brand"))).withColumn('sub_category',trim(col("sub_category"))).withColumn('sku',trim(col("sku"))).withColumn('supplier_id',trim(col("supplier_id"))).withColumn('weight_kg',trim(col("weight_kg"))).withColumn("created_date", to_date(col("created_date"), "yyyy-MM-dd")).withColumn("created_at", to_timestamp(col("created_at"), "yyyy-MM-dd'T'HH:mm:ss"))
df_products=df_products.withColumn('product_id',regexp_replace(col('product_id'),r'[^A-Za-z0-9]',"")).withColumn('brand',regexp_replace(col('brand'),r'[^A-Za-z0-9]',""))


In [0]:
standard_checks(df_products,'products',['product_id','product_name','category','brand'],['product_id'])
write_delta_overwrite(df_products,'/Volumes/salesdw/silver/silver_data/silver_products',['category'])

## **Processing Orders**

In [0]:
df_orders=spark.read.format("delta")\
        .option('inferSchema','true')\
        .load('/Volumes/salesdw/bronze/bronze_data/bronze_orders')

In [0]:
df_products=dedup(df_products,['order_id','customer_id','product_id','order_line_id'],'order_date')

In [0]:
  df_orders=df.orders
        .withColumn("order_id",       upper(trim(col("order_id"))))
        .withColumn("order_line_id",  upper(trim(F.col("order_line_id"))))
        .withColumn("customer_id",    upper(trim(F.col("customer_id"))))
        .withColumn("product_id",     upper(trim(F.col("product_id"))))
        .withColumn("order_date",     to_date(col("order_date"),  "yyyy-MM-dd"))
        .withColumn("ship_date",      to_date(col("ship_date"),   "yyyy-MM-dd"))
        .withColumn("quantity",       col("quantity").cast(IntegerType()))
        .withColumn("unit_price",     col("unit_price").cast(DecimalType(10, 2)))
        .withColumn("discount_pct",   col("discount_pct").cast(DecimalType(5, 4)))
        .withColumn("payment_method", initcap(F.trim(F.col("payment_method"))))
        .withColumn("order_channel",  initcap(F.trim(F.col("order_channel"))))
        .withColumn("order_status",   initcap(F.trim(F.col("order_status"))))
        # Derived metrics
        .withColumn("discount_amount",
                    round(col("unit_price") * col("quantity") * col("discount_pct"), 2))
        .withColumn("gross_revenue",
                    round(col("unit_price") * col("quantity"), 2))
        .withColumn("net_revenue",
                    round(F.col("gross_revenue") - col("discount_amount"), 2))
        .withColumn("days_to_ship",
                    datediff(F.col("ship_date"), col("order_date")))
        .withColumn("created_at", current_timestamp())
        .withColumn("updated_at", current_timestamp())
  

In [0]:
   df_orders = df_orders.filter(
        (F.col("quantity") > 0) &
        (F.col("unit_price") > 0)
    )

In [0]:
 final_cols = [
        "order_id", "order_line_id", "customer_id", "product_id",
        "order_date", "ship_date", "quantity", "unit_price",
        "discount_pct", "discount_amount", "gross_revenue", "net_revenue",
        "days_to_ship", "payment_method", "order_channel", "order_status",
        "source_system", "batch_id", "ingested_at", "created_at", "updated_at"
    ]
    df_orders = df_orders.select(*final_cols)

In [0]:
standard_checks(df_orders,'orders',['order_id','customer_id','product_id','order_line_id'],['order_id'])
write_delta_overwrite(df_orders,'/Volumes/salesdw/silver/silver_data/silver_orders',['order_id'])
logger.info(f" Silver orders written → {SILVER_ORDERS}")